

```
# 器材借用系統
```



In [ ]:
!pip install -q groq gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 3.6 MB/s eta 0:00:00


In [ ]:
# import library
import sqlite3
from google.colab import userdata #important
from groq import Groq
import gradio as gr

In [ ]:
#acces API
client = Groq(api_key = userdata.get('GROQ_API_KEY'))
print('Groq連線成功')

Groq連線成功


In [ ]:
conn = sqlite3.connect('equipment.db')
cursor = conn.cursor()

# 2. 刪除（刪除舊資料表，確保舊的資料已刪除）
cursor.execute('''DROP TABLE IF EXISTS items''')
cursor.execute('''DROP TABLE IF EXISTS borrow_logs''')

# 3. 建立 items table (器材表)
cursor.execute('''
CREATE TABLE items (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT,
    status TEXT
)
''')

# 4. 建立 table: borrow_logs (借用紀錄表)
cursor.execute('''
CREATE TABLE borrow_logs (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    item_name TEXT,
    borrow_time TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
''')

# 5. 插入 item into items: 在資料表items中心新增器材
initial_data = [
    ('麥克風', '可借用'),
    ('筆電', '可借用'),
    ('HDMI', '已借出'),
    ('3D列印機', '已借出')
]
cursor.executemany("INSERT INTO items (name, status) VALUES (?, ?)", initial_data)

# 6. 儲存資料庫 (將所有更新置資料庫內，並關閉資料庫存取)
conn.commit()
conn.close()
print('資料庫建立完成，已建立好items與borrow_logs資料表')

資料庫建立完成，已建立好items與borrow_logs資料表


In [ ]:
# building function with GenAI
def borrow_item(user_input):
  prompt = f'''
  系統目前只有4種器材:[麥克風，筆電，HDMI，3軸穩定器],
  請從學生的話裡面，找出他想要借用哪種器材，
  規則:只回傳器材名稱(例如:麥克風、筆電、HDMI、3軸穩定器)。
  如果學生講的不是這4種,請回傳“未知”。絕對不要有多餘的字。

  學生的話: {user_input}
  '''

  response = client.chat.completions.create(
  model = 'openai/gpt-oss-120b',
  messages = [{'role':'system', 'content' :prompt}],
  temperature=0.0 #請AI完全依照我的指令
  )

  return (response.choices[0].message.content)


In [ ]:
#testing function: borrow_item
print (borrow_item('上課要錄動態影片,我要借3軸穩定器'))

3軸穩定器


In [ ]:
# borrow system
def borrow_system(user_say):
    # 0. 先請AI找出要借用的設備名稱
    target_item = borrow_item(user_say)

    # 1. 如果沒找到，顯示相關訊息
    if target_item not in ['麥克風', '筆電', 'HDMI', '3軸穩定器']:
        return f'抱歉，系統中找不到"{target_item}"這個設備'

    # 2. 如果找到，顯示狀態
    conn = sqlite3.connect('equipment.db')
    cursor = conn.cursor()
    cursor.execute("SELECT status FROM items WHERE name = ?", (target_item,))
    row = cursor.fetchone()

    current_status = row[0]

    print (f'設備"{target_item}"的狀態是"{current_status}"')

    #3.按照狀態給出不同回應
    if current_status == '以借出':
      conn.close()
      return f'抱歉，"{target_item}"已被借出了。'
    else:
      cursor.execute("UPDATE items SET status = '已借出' WHERE name = ?", (target_item,))
      cursor.execute("INSERT INTO borrow_logs (item_name) VALUES (?)", (target_item,))
      conn.commit()
      conn.close()
      return f'已將"{target_item}"借出。'


In [ ]:
#testing function: borrow_item
borrow_system('上課要錄動態影片,需要使用筆電')

設備"筆電"的狀態是"可借用"


'已將"筆電"借出。'

In [ ]:
# 用gradio呈現網頁
def web_interface(message):
    if not message.strip():
        return "輸入想借用什麼器材！"
    reply = borrow_system(message)
    return reply

demo = gr.Interface(
    fn = web_interface,
    inputs = gr.Textbox(label = '請輸入您的器材需求'),
    outputs = gr.Textbox(label = '系統回應'),
    title = '簡易器材借出系統',
    description = 'Groq API + sqlite3 = 器材系統'
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c73b3f3b19e0c69b74.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
